# CEvNS Scattering Cross Section Example

This notebook computes a simple differential scattering cross section in the split minimal copy.

Run it from:

`F:\\Work\\2026-07-15 CEvNSpyEFT-Local\\260715Gitup1`

Main API used here:

- `ResponseDB` loads nuclear response functions from `data/responses/responses_all.json`.
- `differential_cross_section_nn` computes `d sigma / d ER` in `cm^2 / keV`.
- Inputs use recoil energy `ER_keV` and neutrino energy `Enu_MeV`.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists())
SRC = ROOT / "src"
RESPONSES = ROOT / "data" / "responses" / "responses_all.json"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from cevns_pyeft.nuclear.response import ResponseDB
from cevns_pyeft.rates.differential import differential_cross_section_nn

print({"root": str(ROOT), "responses_exists": RESPONSES.exists()})


{'root': 'f:\\Work\\2026-07-15 CEvNSpyEFT-Local\\260715Gitup1', 'responses_exists': True}


## 1. Load Target And Choose Kinematics

This example uses `Cs133`, a fixed neutrino energy of `30 MeV`, and scans recoil energy from `1 keV` to `30 keV`.


In [2]:
db = ResponseDB(RESPONSES)
resp = db.get("Cs133")

Enu_MeV = 30.0
ER_grid_keV = np.linspace(1.0, 30.0, 30)

rows = []
for ER_keV in ER_grid_keV:
    result = differential_cross_section_nn(
        resp,
        ER_keV=float(ER_keV),
        Enu_MeV=Enu_MeV,
        include_sm=True,
        sin2thetaW=0.23873,
        basis="np",
    )
    rows.append({
        "target": resp.name,
        "Enu_MeV": Enu_MeV,
        "ER_keV": float(ER_keV),
        "dsigma_cm2_per_keV": float(result.dsigma_cm2_per_keV),
    })

df = pd.DataFrame(rows)
df.head()


,target,Enu_MeV,ER_keV,dsigma_cm2_per_keV
0,Cs133,30.0,1.0,2.629447e-39
1,Cs133,30.0,2.0,2.307884e-39
2,Cs133,30.0,3.0,2.012152e-39
3,Cs133,30.0,4.0,1.740593e-39
4,Cs133,30.0,5.0,1.491641e-39


## 2. Inspect Values

The values below are in `cm^2 / keV`.


In [3]:
summary = {
    "target": resp.name,
    "points": int(len(df)),
    "min_dsigma_cm2_per_keV": float(df["dsigma_cm2_per_keV"].min()),
    "max_dsigma_cm2_per_keV": float(df["dsigma_cm2_per_keV"].max()),
    "finite": bool(np.isfinite(df["dsigma_cm2_per_keV"].to_numpy()).all()),
    "nonnegative": bool((df["dsigma_cm2_per_keV"].to_numpy() >= 0.0).all()),
}
print(summary)
df.tail()


{'target': 'Cs133', 'points': 30, 'min_dsigma_cm2_per_keV': 0.0, 'max_dsigma_cm2_per_keV': 2.6294471890508138e-39, 'finite': True, 'nonnegative': True}


,target,Enu_MeV,ER_keV,dsigma_cm2_per_keV
25,Cs133,30.0,26.0,0.0
26,Cs133,30.0,27.0,0.0
27,Cs133,30.0,28.0,0.0
28,Cs133,30.0,29.0,0.0
29,Cs133,30.0,30.0,0.0


## 3. Optional: Plot

This plot is only for visual inspection. The calculation above does not depend on plotting.


In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df["ER_keV"], df["dsigma_cm2_per_keV"], marker="o", linewidth=1.5)
ax.set_xlabel("Recoil energy ER [keV]")
ax.set_ylabel("d sigma / d ER [cm^2 / keV]")
ax.set_title(f"CEvNS {resp.name}, Enu = {Enu_MeV:g} MeV")
ax.grid(True, alpha=0.3)
fig.tight_layout()
out_path = ROOT / "cross_section_example.png"
fig.savefig(out_path, dpi=150)
plt.close(fig)
print({"plot": str(out_path), "saved": out_path.exists()})


{'plot': 'f:\\Work\\2026-07-15 CEvNSpyEFT-Local\\260715Gitup1\\cross_section_example.png', 'saved': True}


## 4. Direct Li2026 Cross Section Entry

If you want to bypass the SM helper and provide LEFT coefficients directly, use `dsigma_dER_li2026_cm2_per_GeV`. Its return value is `cm^2 / GeV`, so divide by `1e6` to convert to `cm^2 / keV`.


In [5]:
from cevns_pyeft.theory.li2026.cross_section import dsigma_dER_li2026_cm2_per_GeV

left_hat = {
    (6, 1, "u"): 1.0e-5 + 0j,
    (6, 1, "d"): -2.0e-5 + 0j,
}

direct_cm2_per_GeV = dsigma_dER_li2026_cm2_per_GeV(
    resp,
    E_nu_GeV=Enu_MeV * 1.0e-3,
    ER_keV=10.0,
    left_hat=left_hat,
    basis="np",
)

print({
    "ER_keV": 10.0,
    "Enu_MeV": Enu_MeV,
    "direct_cm2_per_GeV": float(direct_cm2_per_GeV),
    "direct_cm2_per_keV": float(direct_cm2_per_GeV / 1.0e6),
})


{'ER_keV': 10.0, 'Enu_MeV': 30.0, 'direct_cm2_per_GeV': 7.593019747581757e-39, 'direct_cm2_per_keV': 7.593019747581758e-45}
